## [HashiCorp Vault](https://developer.hashicorp.com/vault) (Injecting Variante)

ist eine zentrale Plattform für Secret-Management, Identity und Verschlüsselung in modernen Infrastruktur- und Kubernetes-Umgebungen. 

Vault ermöglicht die sichere Verwaltung von Passwörtern, API-Keys, Zertifikaten und dynamischen Credentials und integriert sich direkt mit Kubernetes, Cloud-Plattformen und CI/CD-Systemen. 

Secrets können entweder als Kubernetes Secrets synchronisiert oder erst zur Laufzeit direkt in Pods injiziert werden. Dadurch eignet sich Vault besonders für Plattformen mit hohen Anforderungen an Security, Auditierung, Rotation und Zero-Trust-Architekturen.

Für Kubernetes existieren verschiedene Integrationsmodelle wie Vault Agent Injector, CSI Driver und Operators. Zusätzlich kann Vault mit GitOps-Workflows kombiniert werden.


Installation von Vault 
* im Dev-Mode
* ohne CSI Driver Support
* ohne produktive Persistenz/TLS/HA
* mit Vault Agent Injector

In [ ]:
%%bash
helm repo add hashicorp https://helm.releases.hashicorp.com
helm repo update

helm install vault hashicorp/vault \
  --namespace vault \
  --create-namespace \
  --set "server.dev.enabled=true" \
  --set "server.dev.devRootToken=root" \
  --set "injector.enabled=true" \
  --set "csi.enabled=false"

Dann Vault konfigurieren:

In [ ]:
%%bash
kubectl exec -n vault vault-0 -- sh -c '
export VAULT_ADDR=http://127.0.0.1:8200
export VAULT_TOKEN=root

vault auth enable kubernetes || true

vault write auth/kubernetes/config \
  kubernetes_host=https://kubernetes.default.svc

vault kv put secret/app username=demo password=supersecret

vault policy write app - <<EOF
path "secret/data/app" {
  capabilities = ["read"]
}
EOF

vault write auth/kubernetes/role/app \
  bound_service_account_names=app \
  bound_service_account_namespaces=default \
  policies=app \
  ttl=1h
'

Dann App-Pod mit Injector-Annotationen:

In [ ]:
%%bash
kubectl apply -f - <<EOF
apiVersion: v1
kind: ServiceAccount
metadata:
  name: app
  namespace: default
---
apiVersion: v1
kind: Pod
metadata:
  name: vault-injector-demo
  namespace: default
  annotations:
    vault.hashicorp.com/agent-inject: "true"
    vault.hashicorp.com/role: "app"
    vault.hashicorp.com/agent-inject-secret-config.txt: "secret/data/app"
spec:
  serviceAccountName: app
  containers:
  - name: app
    image: busybox:1.36
    command: ["sh", "-c", "sleep 3600"]
EOF

Testen

In [ ]:
%%bash
kubectl wait --for=condition=Ready pod/vault-injector-demo --timeout=120s

kubectl exec vault-injector-demo -c app -- ls -la /vault/secrets
kubectl exec vault-injector-demo -c app -- cat /vault/secrets/config.txt

---

### Aufräumen

In [ ]:
%%bash
kubectl delete pod vault-injector-demo --ignore-not-found
kubectl delete serviceaccount app --ignore-not-found
helm uninstall vault -n vault
kubectl delete ns vault